# Orchestration with Swarm

## What's covered

- **Why orchestration** — the wall single-host Docker (and Compose) hits, and what an orchestrator does for you
- **Swarm vs Kubernetes** — what each is good at, and why this notebook covers Swarm
- **Swarm architecture** — managers, workers, the raft consensus log, and the gossip plane
- **Initialising a cluster** — `docker swarm init`, `docker swarm join`, manager and worker tokens
- **Services, not containers** — the orchestrator's unit of work, replicated vs global mode
- **The service lifecycle** — `create`, `ls`, `ps`, `inspect`, `update`, `scale`, `rollback`, `rm`
- **Rolling updates** — `--update-parallelism`, `--update-delay`, `--update-failure-action`
- **Overlay networks** revisited — multi-host networking, swarm-attachable networks
- **The ingress network and routing mesh** — how a single published port reaches the right replica on any node
- **Stacks** — `docker stack deploy` and the Compose-flavoured stack file
- **Service discovery and DNS** — virtual IPs (VIP) and DNS round-robin
- **Swarm secrets and configs** — encrypted-at-rest credentials and mounted config files
- **Placement** — constraints, preferences, labels, max-replicas-per-node
- **Node management** — `drain`, `pause`, `active`, `promote`, `demote`, taking nodes down for maintenance
- **The realities** — when Swarm is the right answer in 2026, and when Kubernetes is

By the end you should be able to bootstrap a three-node Swarm, deploy a multi-service stack across it, roll it forward and back without downtime, and drain a node for maintenance — all from a single CLI.

## Why orchestration

Compose got us to "a multi-container app on one host." That's already useful, but it hits a wall:

- **What if one host isn't enough capacity?**
- **What happens if that host dies?** Compose has no answer — your stack is down until you rebuild the host.
- **How do you do a rolling deploy** without taking traffic offline?
- **How do you keep secrets** off disk on every machine that runs the service?
- **How do you have services on different hosts** talk to each other by hostname?

An **orchestrator** answers all five. It manages a pool of hosts (a **cluster**), accepts a *desired state* ("I want 3 replicas of this service everywhere"), and continuously reconciles reality to that state. A node dies; the orchestrator launches replacement replicas elsewhere. A new image is deployed; replicas roll forward one at a time. A service publishes a port; the orchestrator routes traffic to whichever node has a healthy replica.

Two orchestrators matter for the Docker world: **Docker Swarm** (built into Docker, simpler, what the DCA exam tests) and **Kubernetes** (more powerful, more complex, the industry default for non-trivial production).

## Swarm vs Kubernetes

| Dimension | Swarm | Kubernetes |
|---|---|---|
| **Installation** | `docker swarm init` — one command, zero extra software | `kubeadm`, managed services (EKS/GKE/AKS), or 5+ binaries. Nontrivial. |
| **CLI** | `docker` — same client you already know | `kubectl` — entirely separate model |
| **Concepts** | Service, task, stack, secret, config | Pod, Deployment, ReplicaSet, Service, Ingress, ConfigMap, Secret, PV, PVC, StatefulSet, DaemonSet, Job, CronJob, Namespace, RBAC, NetworkPolicy, CRDs... |
| **Networking** | Built-in overlay, routing mesh, simple | CNI plugins; networking is your choice (Calico, Cilium, etc) |
| **Storage** | Local volumes + 3rd-party drivers | CSI ecosystem; cloud-native storage |
| **Updates** | Rolling, with rollback | Rolling, blue-green, canary, with controllers for everything |
| **Ecosystem** | Small, mostly Docker-shipped | Vast — Helm, operators, service meshes, GitOps tools |
| **Multi-cluster / federation** | None | Several solutions, all complicated |
| **Pace of development** | Mature, mostly stable | Rapidly evolving |

Both run OCI images. The image you built in notebook 02 deploys identically on either.

**Swarm is the right answer when:** you want orchestration without the Kubernetes complexity tax. Small teams, edge/IoT clusters, simple production setups, learning orchestration concepts. **The DCA exam tests Swarm** — Kubernetes is out of scope.

**Kubernetes is the right answer when:** you need its ecosystem (operators, service meshes, GitOps), or you're going multi-cluster, or you're hiring at scale (the talent pool knows Kubernetes). Most large companies are on Kubernetes today.

Despite the industry trend, learning Swarm is *not* wasted time. The concepts (declarative state, rolling updates, secrets, placement) carry over directly. Kubernetes is "Swarm but bigger, harder, with more knobs." Start here.

## Swarm architecture

A Swarm cluster is a set of Docker hosts (**nodes**) talking to each other. Each node plays one of two roles:

- **Manager** — runs the orchestrator's control plane: accepts API calls (`docker service create ...`), schedules tasks, stores cluster state. Managers also run workloads by default (you can prevent this).
- **Worker** — runs containers (called **tasks** in Swarm vocab) assigned by managers. Does not participate in control plane decisions.

You can have 1, 3, 5, or 7 managers (always odd, for raft quorum). The cluster tolerates `(N-1)/2` manager failures — 3 managers tolerate 1 loss, 5 tolerate 2. **One** worker count is unlimited.

```
                       +--------------------+
                       | Manager #1 (leader)|
                       +--------+-----------+
                                |
                          raft  |  raft
                                v
            +-------------+   +-------------+   +-------------+
            | Manager #2  |   | Manager #3  |   |  Worker #1  |
            +-------------+   +-------------+   +-------------+
                                                      |
                                                  +---+----+
                                                  |        |
                                              +---v----+ +-v-----+
                                              |Worker#2| |Worker#3|
                                              +--------+ +-------+
```

Three protocols glue the cluster together:

- **Raft** — a leader-election + log-replication protocol among managers. The leader writes cluster state (services, tasks, secrets, networks) to a replicated raft log. Reads are cheap; writes go through the leader.
- **Gossip** — a peer-to-peer protocol every node speaks. Tells everyone "service `web` has tasks on nodes A, C, D at these IPs." Used for service discovery and overlay-network membership.
- **Docker API** — managers expose the same REST API the daemon does, just with extra Swarm verbs (`service`, `node`, `stack`, `secret`, `config`, `network` with overlay driver).

## Initialising a cluster

On the machine that will be your first manager:

```bash
docker swarm init --advertise-addr <THIS-MACHINE-IP>
```

That single command:

1. Turns the local Docker daemon into a Swarm manager.
2. Generates two join tokens — one for additional **managers**, one for **workers**.
3. Initialises the raft log with this node as leader.
4. Creates the default overlay network (`ingress`).
5. Prints the worker-join command, ready to paste on another machine.

You'll see something like:

```
To add a worker to this swarm, run the following command:

    docker swarm join --token SWMTKN-1-xxxx... <THIS-MACHINE-IP>:2377
```

On each additional node:

```bash
# Worker
docker swarm join --token SWMTKN-1-worker-... <MANAGER-IP>:2377

# Another manager (re-print the manager token on the existing manager)
docker swarm join --token SWMTKN-1-manager-... <MANAGER-IP>:2377
```

To re-print tokens later:

```bash
docker swarm join-token worker
docker swarm join-token manager
docker swarm join-token --rotate manager   # invalidate old token, issue new
```

The cluster runs on **port 2377** (cluster management, TCP), **7946** (gossip, TCP+UDP), and **4789** (overlay data, UDP). Open those between nodes.

```bash
docker node ls   # see all nodes; * marks the one you're on
```

## Services, not containers

The Swarm primitive isn't a container; it's a **service**. A service is a declaration:

> "I want N replicas of image X running across the cluster, with these networks, secrets, env vars, resources, and update policy. Make it so."

The orchestrator decomposes a service into **tasks** (one per replica), schedules tasks onto nodes, and creates containers to run them. The container is just an implementation detail.

```bash
docker service create --replicas 3 --name web -p 8080:80 nginx:alpine
docker service ls
# ID            NAME    MODE         REPLICAS   IMAGE          PORTS
# r4f...        web     replicated   3/3        nginx:alpine   *:8080->80/tcp

docker service ps web    # see the individual tasks, which nodes they're on
```

If one of those nodes crashes, the manager notices, re-schedules its task elsewhere, and `replicas 3/3` is restored automatically.

### Two service modes

- **Replicated** (default) — N tasks, scheduler picks where. The 90% case.
- **Global** — exactly one task per node. Used for per-node sidecars: log shippers, monitoring agents, security tools.

```bash
docker service create --mode global --name node-exporter prom/node-exporter
```

`docker service ps node-exporter` will show one task per node, automatically growing if you join more nodes.

## The service lifecycle

The Swarm equivalents of the `docker run`/`stop`/etc commands work at the service level.

| Command | What it does |
|---|---|
| `docker service create [flags] IMAGE [CMD]` | Create a new service. Most `docker run` flags apply (`-p`, `-e`, `-v`, `--mount`, `--user`, etc) plus Swarm-specific ones. |
| `docker service ls` | List services with desired vs running replicas. |
| `docker service ps SERVICE` | Tasks in this service, with node placement and status. |
| `docker service inspect SERVICE` | Full JSON definition. `--pretty` for human-readable. |
| `docker service logs [-f] SERVICE` | Aggregated logs from every task in the service. |
| `docker service update [flags] SERVICE` | Change a service's definition (image, replicas, env, networks, ...). Triggers a rolling update. |
| `docker service scale SERVICE=N [...]` | Shortcut for `update --replicas`. |
| `docker service rollback SERVICE` | Revert the last update. |
| `docker service rm SERVICE` | Delete the service. All its tasks/containers go away. |

Two important conceptual shifts from single-host Docker:

1. **You don't stop or start a service.** You scale it to 0 to "stop" (it'll come back to N on the next scale-up); you `rm` it to remove.
2. **You don't restart a container manually.** If a task dies, the orchestrator replaces it. Manual intervention is for code/config changes, not "kick the tires."

## Rolling updates and rollback

A service update is one of the most common day-to-day Swarm operations. The default behaviour:

```bash
docker service update --image myorg/api:1.2.4 api
```

Swarm updates one task at a time: stop one old task, start one new task, wait for it to be ready, move to the next. Default delay between batches is 0; batch size 1.

Tune with:

```bash
docker service update \
  --image myorg/api:1.2.4 \
  --update-parallelism 2 \
  --update-delay 10s \
  --update-failure-action rollback \
  --update-monitor 30s \
  --update-max-failure-ratio 0.2 \
  api
```

What each flag means:

- **`--update-parallelism N`** — update N tasks at a time.
- **`--update-delay 10s`** — pause this long between batches.
- **`--update-monitor 30s`** — after starting a new task, watch it for this long before considering it "successful." Catches "starts fine, crashes after 20s" bugs.
- **`--update-max-failure-ratio 0.2`** — abort if more than 20% of tasks fail their monitor window.
- **`--update-failure-action rollback`** — on abort, automatically revert to the previous image. Other options: `continue`, `pause`.

These can also be **set at service creation** so they apply to future updates without re-specifying:

```bash
docker service create \
  --update-parallelism 2 --update-delay 10s --update-failure-action rollback \
  --name api myorg/api:1.2.3
```

### Manual rollback

If something went wrong:

```bash
docker service rollback api
```

This reverts to the previous spec. Swarm keeps the previous one revision in history (configurable via `--rollback-*` flags symmetric to `--update-*`).

## Overlay networks revisited

Notebook 05 introduced overlay networks as "the multi-host bridge." In Swarm they're how every service communicates across nodes.

```bash
docker network create --driver overlay --attachable app-net
```

- **`--driver overlay`** — required for multi-host.
- **`--attachable`** — allows standalone containers (not just Swarm services) to join. Optional but commonly useful for debugging.

An overlay uses **VXLAN** under the hood: each packet between containers on different hosts is wrapped in a UDP datagram that traverses port 4789 between nodes. To the containers it looks like a flat LAN.

```bash
docker service create --network app-net --name api myorg/api
docker service create --network app-net --name web --publish 8080:80 myorg/web
```

`web` and `api` are now on the same logical network. `web` reaches `api` by service name (`http://api`) — Swarm runs the embedded DNS resolver from notebook 05, but cluster-wide.

Two important nuances:

- **Tasks of the same service share a VIP.** `api` resolves to a single virtual IP. The kernel's IPVS load-balances connections across all healthy `api` tasks, regardless of which node they're on.
- **Encryption is opt-in.** Add `--opt encrypted` to encrypt traffic between nodes on this network (IPsec). On by default for the `ingress` network.

## The ingress network and routing mesh

A killer Swarm feature, hidden behind a benign-looking flag.

When a service publishes a port (`docker service create -p 8080:80 ...`), Swarm puts that port on **every node in the cluster**. Even nodes that aren't running a replica of the service.

A request hitting any node:port reaches a healthy replica somewhere, even if that node hosts no replicas itself. The mechanism:

1. The node receives the packet on its `:8080`.
2. The kernel's IPVS, configured by Swarm, picks a healthy task IP (across the cluster).
3. The packet is routed (over the encrypted `ingress` overlay) to the chosen task's node.
4. That node delivers to the local container.

```
  Internet ----> Node A:8080  ---routing mesh-->  task running on Node C
            ----> Node B:8080  ---routing mesh-->  task running on Node A
            ----> Node C:8080  --local->            task running on Node C
```

Put **any node's IP** behind your DNS or load balancer and it works. The cluster handles internal routing. This is the **routing mesh** — the default behaviour for `-p` in Swarm.

If you don't want this (you want the port published only on nodes actually running the task), use **host mode**:

```bash
docker service create -p mode=host,published=8080,target=80 --name web myorg/web
```

Host mode skips the mesh; useful when an external load balancer is already smart (the cloud LB pinging healthchecks).

## Stacks — the Compose file in cluster form

Typing 15-flag `docker service create` lines is the Swarm equivalent of typing `docker run` lines instead of using Compose. The fix: **stacks**.

A stack is a YAML file in the Compose format (with extra `deploy:` keys), deployed to a Swarm with:

```bash
docker stack deploy -c stack.yaml my-app
```

```yaml
# stack.yaml
services:
  web:
    image: myorg/web:1.2.3
    ports:
      - "8080:80"
    networks: [app-net]
    deploy:
      replicas: 3
      update_config:
        parallelism: 1
        delay: 10s
        failure_action: rollback
        monitor: 30s
      restart_policy:
        condition: on-failure
      resources:
        limits:
          cpus: "0.5"
          memory: 256M
      placement:
        constraints:
          - node.role == worker

  api:
    image: myorg/api:1.4.5
    networks: [app-net]
    secrets: [db_password]
    deploy:
      replicas: 5
      placement:
        max_replicas_per_node: 2

  db:
    image: postgres:16
    volumes:
      - pgdata:/var/lib/postgresql/data
    networks: [app-net]
    secrets: [db_password]
    deploy:
      replicas: 1
      placement:
        constraints:
          - node.labels.storage == ssd

networks:
  app-net:
    driver: overlay

volumes:
  pgdata: {}

secrets:
  db_password:
    file: ./secrets/db_password.txt
```

Then:

```bash
docker stack deploy -c stack.yaml my-app
docker stack services my-app
docker stack ps my-app
docker stack rm my-app
```

Note the `deploy:` block — *that's the bit Compose ignores on a single host and Swarm pays attention to*. `replicas`, `update_config`, `placement`, `resources` all live there. Outside `deploy:`, the file is the same Compose syntax.

## Service discovery and DNS

A service named `api` is reachable on the overlay network by the hostname `api`. There are two flavours of resolution:

**Virtual IP (VIP) mode** (default) — `api` resolves to a single, stable virtual IP. The kernel load-balances incoming connections across all healthy tasks (using IPVS). Clients see one IP and don't need to know how many replicas there are.

**DNS round-robin (DNSRR) mode** — `api` resolves to the IPs of *all* current tasks. Each DNS lookup returns a different one (round-robin). The client picks; if a task dies, the client may hit a stale IP and retry.

Switch with:

```yaml
services:
  api:
    image: myorg/api
    deploy:
      endpoint_mode: dnsrr   # or "vip", the default
```

**Use VIP** for almost everything — it's stable, kernel-fast, and tasks coming/going is transparent. **Use DNSRR** only for clients that load-balance themselves and want raw target IPs (some service meshes, custom gossip protocols).

For service-to-service, Swarm's DNS resolves only **within the same overlay network**. A service in network `frontend` can't resolve a service only on `backend` — by design.

## Swarm secrets and configs

From notebook 08, secrets-as-files. In Swarm they're a first-class object stored encrypted at rest in the raft log.

```bash
echo "hunter2" | docker secret create db_password -
docker secret ls
docker secret inspect db_password   # metadata; value is not returned
```

Attach to a service:

```bash
docker service create \
  --secret db_password \
  --name api myorg/api
```

Inside the `api` container, the file appears at `/run/secrets/db_password` (override with `target=...`), permissions `0444` owned by root. Only nodes running an `api` task ever see the secret — managers don't push secrets to nodes that don't need them.

**Configs** are the same idea but for non-secret config files:

```bash
docker config create nginx_conf ./nginx.conf
docker service create \
  --config source=nginx_conf,target=/etc/nginx/nginx.conf \
  --publish 80:80 --name nginx nginx:alpine
```

The config is distributed to every node running a task and mounted at the target path. Update by removing the old config + creating a new one + updating the service — Swarm doesn't (currently) version configs in place.

**Both secrets and configs are immutable.** To change a value, create a new object with a new name and update the service to reference it. This is by design: rolling out a config change is a service update, with all the rolling-update guarantees that brings.

## Placement — putting tasks on the right nodes

Sometimes you want a task on a *specific* class of node. Examples: the DB must be on a node with SSDs; the legacy app must be on a Linux-amd64 node; you want at most 2 replicas per node so a node failure doesn't take all of them.

### Constraints — hard requirements

```yaml
deploy:
  placement:
    constraints:
      - node.role == worker            # only on worker nodes
      - node.labels.storage == ssd     # only on nodes you've labelled
      - node.platform.os == linux
      - node.platform.arch == arm64
```

`docker node update --label-add storage=ssd nodeX` to add a label to a node.

### Preferences — soft hints

```yaml
deploy:
  placement:
    preferences:
      - spread: node.labels.datacenter   # spread tasks evenly across datacenters
```

### Replica caps

```yaml
deploy:
  replicas: 9
  placement:
    max_replicas_per_node: 2   # at most 2 replicas of this service on any one node
```

With 9 replicas and `max_replicas_per_node: 2`, you need at least 5 eligible nodes.

## Node management

`docker node ls` shows every node and its **availability** state.

- **`active`** — accepting new tasks. Default.
- **`pause`** — runs existing tasks but won't accept new ones.
- **`drain`** — won't accept new tasks **and** Swarm re-schedules existing tasks off the node. Used to take a node down for maintenance without dropping replicas.

```bash
docker node update --availability drain nodeX
# wait until docker node ps nodeX is empty
# do maintenance (reboot, kernel upgrade)
docker node update --availability active nodeX
```

This is the "rolling host upgrade" workflow. Combined with rolling-update services, you can take a whole cluster through a kernel/OS upgrade without taking traffic offline.

### Promote / demote

Add or remove a node from the manager set without rebuilding:

```bash
docker node promote workerX    # add to the manager set
docker node demote managerX    # back to worker only
```

You'd promote to grow the manager count (e.g. from 1 to 3 for HA); demote when scaling back or before removing a node.

### Removing a node

```bash
# On the node itself, leave the swarm
docker swarm leave            # for workers
docker swarm leave --force    # for managers (be careful — quorum implications)

# On a manager, clean up the entry afterwards
docker node rm nodeX
```

## The realities — Swarm in 2026

Be honest about where Swarm stands.

**What works well:**

- Setup speed is unbeatable. `docker swarm init` + one `join` per node and you have a cluster in two minutes.
- The mental model is small and stays close to single-host Docker.
- For small-to-medium deployments — a handful of services, single-region, modest scale — Swarm does the job with a fraction of the operational overhead of Kubernetes.
- Edge and IoT clusters frequently land on Swarm or k3s for the same reason.

**What doesn't:**

- The ecosystem has stagnated since Kubernetes won. New tools (operators, service meshes, GitOps controllers) almost always target Kubernetes first; Swarm support is grudging or absent.
- Swarm's update model is fine but lacks the richness of Kubernetes' (no Deployment vs StatefulSet distinction, simpler health-checks, no native canary or blue-green).
- Hiring on "Swarm experience" in 2026 is a smaller pool than on Kubernetes.

**For the DCA exam** — Swarm is core content. Learn it well, including the flags above, the routing mesh, secrets, and placement constraints.

**For your career** — knowing both is valuable. Swarm clarifies the *concepts* that Kubernetes obscures behind abstractions. After you can comfortably operate a Swarm, the Kubernetes vocabulary maps cleanly:

| Swarm | Kubernetes |
|---|---|
| Service | Deployment + Service |
| Task | Pod |
| Stack | Helm chart / kustomization |
| Routing mesh | kube-proxy / Service of type LoadBalancer |
| Overlay | CNI network |
| Manager | Control plane (API server, scheduler, etcd) |
| Secret / Config | Secret / ConfigMap |
| Placement constraint | nodeSelector / affinity |
| Drain | `kubectl drain` |

Same shapes, different syntax. That's why notebook 09 in this curriculum stands on its own — the concepts will follow you into whatever orchestrator your next job uses.